# CoDA-GQA-L Training

Fine-tune a Llama-family model with CoDA differential attention.

**Runtime**: Change to GPU (A100/H100) via Runtime > Change runtime type.

In [ ]:
!git clone https://github.com/anthony-maio/CoDA-GQA-L.git
%cd CoDA-GQA-L
!pip install -e . transformers datasets -q

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Quick smoke test (SmolLM2-135M, ~5 min)

Phase 1: 100 steps unbounded + Phase 2: 50 steps bounded

In [ ]:
!python benchmarks/train_coda.py \
    --model HuggingFaceTB/SmolLM2-135M \
    --max-steps 100 --bounded-steps 50 --bounded-config medium \
    --eval-every 50 \
    --batch-size 2 --grad-accum 2 --seq-len 512 \
    --freeze coda-only --dataset wikitext-2

## Real run: Mistral 7B with Phase 2

Phase 1: 2000 steps unbounded (learn CoDA differential attention)
Phase 2: 1000 steps bounded (learn to work within limited KV cache)

~6 hours on A100/H100.

In [ ]:
!python benchmarks/train_coda.py \
    --model mistralai/Mistral-7B-v0.3 \
    --max-steps 2000 --bounded-steps 1000 --bounded-config medium \
    --eval-every 200 --save-every 500 \
    --batch-size 2 --grad-accum 4 --seq-len 2048 \
    --freeze attention --lr 5e-5 --lr-coda 1e-3

## Evaluate trained model

In [ ]:
# Find the latest run directory
import glob, os
runs = sorted(glob.glob("runs/Mistral-7B*"))
if runs:
    latest = runs[-1]
    best = os.path.join(latest, "best")
    print(f"Latest run: {latest}")
    print(f"Best checkpoint: {best}")
else:
    print("No Mistral runs found. Run training first or adjust the glob.")
    latest = best = None

In [ ]:
if best:
    !python benchmarks/eval_llm.py \
        --model mistralai/Mistral-7B-v0.3 \
        --experiment perplexity \
        --adapter-weights {best} \
        --head-norm-mode identity \
        --dtype bf16 \
        --bounded-configs medium

## View training log

In [ ]:
import json
if latest:
    log = json.loads(open(os.path.join(latest, "training_log.json")).read())
    losses = [(e["step"], e["loss"]) for e in log if "loss" in e]
    evals = [(e["step"], e["eval_ppl"]) for e in log if "eval_ppl" in e]
    bounded = [e for e in log if "bounded_ppl" in e]

    print(f"Steps: {losses[0][0]} -> {losses[-1][0]}")
    print(f"Loss:  {losses[0][1]:.4f} -> {losses[-1][1]:.4f}")
    if evals:
        print(f"PPL:   {evals[0][1]:.2f} -> {evals[-1][1]:.2f}")
    if bounded:
        print(f"Bounded PPL: {bounded[-1]['bounded_ppl']:.2f}")